In [ ]:
%matplotlib ipympl
RSTEP = 0.01
QMAX = 25.0


In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact
import numpy as np

from diffpy.srreal.pdfcalculator import PDFCalculator, DebyePDFCalculator
from diffpy.structure import loadStructure
from pyobjcryst.crystal import CreateCrystalFromCIF

stru = loadStructure('cif/BaBiO3.cif')
# stru.Uisoequiv = 0.007

def config_pc():
    pc = PDFCalculator()
    pc.qmin, pc.qmax = 0.1, QMAX
    pc.rmin, pc.rmax = 0.0, 20.0
    pc.rstep = RSTEP
    pc.qdamp = 0.06
    # pc.qstep = 0.01
    # pc.debyeprecision = 1.0e-8
    pc.delta2 = 7.51

    return pc


In [ ]:
datao = np.load("data/original.npz")

print("npz keys:", list(datao.keys()))

original_partials = {
    tuple(key[:-4].split('-')): arr
    for key, arr in datao.items()
    if key.endswith('-pdf') and len(key[:-4].split('-')) == 2
}


In [ ]:
import numpy as np

def reconstruct_pdf(pdf, rstep=RSTEP, qmax=QMAX, tlen=65536, q_low_cut=0.2):
    """
    Rebuild G(r) from an input PDF by:
      1) zero-padding G(r) to length tlen,
      2) computing F(Q) = imag(ifft(padded_G)) * rmax (FFT-based sine transform),
      3) zeroing low-Q (<= q_low_cut) and > Nyquist components,
      4) computing G(r) = (2/pi) * sum_k F(Q_k) * sin(Q_k r) * ΔQ,
      5) trimming back to the original length of pdf.

    Parameters
    ----------
    pdf : (N,) array_like
        Input G(r) samples on a uniform grid.
    rstep : float
        Δr for the input G(r).
    qmax : float
        Maximum Q to keep in the reconstruction.
    tlen : int, optional
        Zero-padded FFT length (power of two recommended). Must be ≥ len(pdf).
    q_low_cut : float, optional
        Low-Q mute threshold (Å⁻¹). Set 0.0 to disable.

    Returns
    -------
    gr : (N,) ndarray
        Reconstructed/filtered G(r) on the original r-grid.
    """
    pdf = np.asarray(pdf, dtype=float)
    N = pdf.size
    if tlen < N:
        raise ValueError(f"tlen ({tlen}) must be ≥ len(pdf) ({N}).")

    # ---- G(r) -> F(Q) via FFT-based sine transform
    yy = np.concatenate([pdf, np.zeros(tlen - N, dtype=float)])
    rmax = rstep * tlen
    fq_full = np.imag(np.fft.ifft(yy)) * rmax

    qstep = 2.0 * np.pi / rmax
    # Low-Q mute and >Nyquist removal
    if q_low_cut > 0.0:
        i_low = int(q_low_cut / qstep) + 1
        fq_full[:i_low] = 0.0
    fq_full[tlen // 2:] = 0.0  # drop second half (above Nyquist)

    # Truncate at qmax
    imax = min(int(qmax / qstep) + 1, tlen // 2)
    q = np.arange(imax, dtype=float) * qstep
    fq = fq_full[:imax]

    # ---- F(Q) -> G(r) via discrete sine integral
    r = np.arange(N, dtype=float) * rstep
    gr = (2.0 / np.pi) * (fq @ np.sin(np.outer(q, r))) * qstep

    return gr


In [ ]:
elements = sorted({a.element for a in stru})
pc = config_pc()
r_tot, G_tot = pc(stru)

def pair_partial(A, B):
    pc.maskAllPairs(False)
    pc.setTypeMask(A, B, True)
    if A != B:
        pc.setTypeMask(B, A, True)
    r, G = pc(stru)
    G = reconstruct_pdf(G)
    return r, G

partials = {}
for A in elements:
    for B in elements:
        r, G = pair_partial(A, B)
        partials[(A, B)] = G


In [ ]:
def deweight_partials(partials, c, q):
    """
    De-weight srreal partials
      new = (w_ab * curve) / [c_a c_b f_a(0) f_b(0) / <f(0)>^2] / (2 if a!=b else 1)

    Parameters
    ----------
    partials : dict
        {(A, B): np.ndarray} where values are weighted curves (e.g., w_ab * G_ab or w_ab * F_ab).
        Keys A, B are element symbols like 'Ba', 'Bi', 'O'.
    c : dict[str, float or int]
        Composition (counts or fractions). Will be normalized to fractions internally.
        Example: {'Ba': 1, 'Bi': 1, 'O': 3}
    q : np.ndarray
        Q-grid used for scattering factors. Only q[0] is used.

    Returns
    -------
    dict[(str, str), np.ndarray]
        De-weighted curves with the same keys as `partials`.
    """
    # normalize composition to atomic fractions
    total = float(sum(c.values()))
    cf = {el: float(v) / total for el, v in c.items()}

    # build f0(Q) for each element and <f(Q)>
    elements = sorted(cf.keys())
    f = {el: np.asarray(fxrayatq(el, q), dtype=float) for el in elements}
    favg = np.zeros_like(q, dtype=float)
    for el in elements:
        favg += cf[el] * f[el]

    favg0 = float(favg[0])
    if not np.isfinite(favg0) or favg0 == 0.0:
        raise ZeroDivisionError("Non-finite <f(0)>; check composition or fxrayatq output.")

    # apply your scalar coefficient at Q[0] (+ /2 for off-diagonals)
    out = {}
    for (A, B), arr in partials.items():
        if A not in cf or B not in cf:
            raise KeyError(f"Missing composition for element(s): {A}, {B}")
        fa0 = float(f[A][0]); fb0 = float(f[B][0])
        w0 = (cf[A] * cf[B] * fa0 * fb0) / (favg0 * favg0)
        if not np.isfinite(w0) or w0 == 0.0:
            raise ZeroDivisionError(f"Non-finite weight at Q=0 for pair {A}-{B}")
        x = np.asarray(arr, dtype=float) / w0
        if A != B:
            x = x / 2.0
        out[(A, B)] = x

    return out


q = pc.qgrid
c = {'Ba': 1, 'Bi': 1, 'O': 3}
from diffpy.pdfgetx.cromermann import fxrayatq
partials_deweighted = deweight_partials(partials, c, q)


In [ ]:
def plot_pair(A, B):
    key = (A, B)
    G = partials_deweighted[key]
    plt.figure()
    plt.plot(r_tot, G, label=f'Partial {key[0]}-{key[1]}')
    plt.plot(datao['r'][:2000], datao[f'{key[0]}-{key[1]}-pdf'][:2000], label=f'ADPDFo {key[0]}-{key[1]}', linestyle='dashed')
    plt.xlabel("r")
    plt.ylabel("G")
    plt.legend()
    plt.show()

interact(plot_pair, A=elements, B=elements);


fig, axs = plt.subplots(3, 3, figsize=(12, 12))
for i, A in enumerate(elements):
    for j, B in enumerate(elements):
        key = (A, B)
        ax = axs[i, j]
        G = partials_deweighted[key]
        original_G = original_partials[key]
        pdf_key = f'{A}-{B}-pdf'
        ax.plot(r_tot, G, label=f'Diff {A}-{B}')
        ax.plot(datao['r'][:2000], original_G[:2000], label=f'Paper {A}-{B}', linestyle='dashed')
        ax.set_xlabel('r')
        ax.set_ylabel('G')
        ax.legend(fontsize='small')
fig.tight_layout()
plt.show()


In [ ]:
def build_element_dpdf(partials_pdf, c, q, G_total):
    """
    Build element-centric differentials D_a(r) from deweighted partials G_ab(r):
        D_a(r) = sum_b [ (c_b f_b(0) / <f(0)>) * G_ab(r) ]
    where <f(0)> = sum_x c_x f_x(0), and G_ab(r) is the deweighted (structural) partial.

    Parameters
    ----------
    partials_pdf : dict[(str,str), np.ndarray]
        Deweighted partial PDFs, keys like ('Ba','O'). Either (a,b) or (b,a) may appear.
    c : dict[str, float|int]
        Composition (counts or fractions). Will be normalized.
    q : np.ndarray
        Q grid used for fxrayatq; only q[0] is used.

    Returns
    -------
    dict[str, np.ndarray]
        Mapping element 'a' -> D_a(r).
    """
    # normalize composition to fractions
    tot = float(sum(c.values()))
    cf = {el: float(v) / tot for el, v in c.items()}
    elements = sorted(set(cf.keys()) | {e for ab in partials_pdf for e in ab})

    # f0(0) and <f(0)>
    f0 = {el: float(fxrayatq(el, q)[0]) for el in elements}
    favg0 = sum(cf.get(el, 0.0) * f0[el] for el in elements)
    if not np.isfinite(favg0) or favg0 == 0.0:
        raise ZeroDivisionError("Non-finite <f(0)>. Check composition or fxrayatq.")

    gamma = {b: (cf.get(b, 0.0) * f0[b]) / favg0 for b in elements}

    def get_pair(a, b):
        if (a, b) in partials_pdf: return partials_pdf[(a, b)]
        if (b, a) in partials_pdf: return partials_pdf[(b, a)]
        raise KeyError(f"Missing deweighted partial for pair {a}-{b}")

    G_a = {}
    for a in elements:
        acc = None
        for b in elements:
            term = gamma[b] * get_pair(a, b)
            acc = term if acc is None else acc + term
        G_a[a] = acc

    Gtot = None
    for i, A in enumerate(elements):
        for B in elements[i:]:
            w0 = (cf[A] * cf[B] * f0[A] * f0[B]) / (favg0 * favg0)
            factor = 2.0 if A != B else 1.0
            term = factor * w0 * get_pair(A, B)
            Gtot = term if Gtot is None else Gtot + term
    plt.figure()
    plt.plot(Gtot, label='Reconstructed Gtot')
    plt.plot(G_total, label='Input Gtot', linestyle='dashed')
    plt.legend()
    plt.show()

    out = {}
    for a in elements:
        ga = gamma[a]
        if 1.0 - ga == 0.0:
            raise ZeroDivisionError(f"Cannot build '-{a}' for a single-component system.")
        D_non = (Gtot - ga * G_a[a]) / (1.0 - ga)
        out[a] = G_a[a]
        out[f"-{a}"] = D_non

    return out


oBipdf  = build_element_dpdf(original_partials, c, q, datao['total-pdf'])['Bi']
oNBipdf = build_element_dpdf(original_partials, c, q, datao['total-pdf'])['-Bi']
ototpdf = datao['total-pdf']
G_a = build_element_dpdf(partials_deweighted, c, q, G_tot)['Bi']
G_na = build_element_dpdf(partials_deweighted, c, q, G_tot)['-Bi']


In [ ]:
plt.figure()
plt.plot(r_tot, G_a, label='Bi', color='b')
# plt.plot(datao['r'][:2000], oBipdf[:2000], 'b--', label='paper Bi')

plt.plot(r_tot, G_na, label='-Bi', color='g')
# plt.plot(datao['r'][:2000], oNBipdf[:2000], 'g--', label='paper -Bi')

plt.plot(r_tot, G_tot, label='Gtot', color='r')
# plt.plot(datao['r'][:2000], ototpdf[:2000], 'r--', label='paper Gtot')
plt.xlabel("r")
plt.ylabel("G")
plt.legend()
plt.show()


In [ ]:

np.savez('dpdf.npz', r_tot=r_tot, G_tot=G_tot, G_a=G_a, G_na=G_na)
